# Лабораторная работа №4. Вычисления на ассемблере

**Выполнил:** Рулев Константин Александрович  
**Вариант:** 16

**Цель:** написать программы с вводом числа с клавиатуры и проверить результаты на языке C.

Отчёт содержит все исходники. Для запуска в Google Colab выберите обычную CPU-среду и нажмите «Среда выполнения → Выполнить все».

Результаты получены при выполнении в Linux x86-64. Все файлы для повторного запуска встроены в отчёт.


## 1. Задания варианта


**Задание 6.** Посчитать числа от 1 до n, которые делятся на 5, но не делятся ни на 3, ни на 7.

**Задание 8.** Вставить нуль после каждой цифры числа n. Например, 123 → 102030.

**Задание 13.** Найти числа не больше n, которые делятся на каждую из двух последних цифр.

Вводится целое неотрицательное число без знака и пробелов. Для задач 6 и 13: **0 ≤ n ≤ 1 000 000**. Для задачи 8: **0 ≤ n ≤ 999 999 999**. Эти границы исключают переполнение и слишком долгий перебор. При неверном вводе программа выводит `Invalid input` и завершается с кодом 1.

In [1]:
import random

variant = 16
random.seed(variant)
print('Задачи:', [i + 1 for i in random.sample(range(14), 3)])

Задачи: [6, 8, 13]


## 2. Подготовка среды

Нужна Linux x86-64. Эта ячейка создаёт рабочую папку и при необходимости устанавливает инструменты сборки.

In [2]:
import os
import platform
import shutil
import subprocess
from pathlib import Path

if platform.system() != 'Linux' or platform.machine() not in ('x86_64', 'amd64'):
    raise RuntimeError('Выберите CPU-среду Linux x86-64 в Google Colab')
if not all(shutil.which(name) for name in ('fasm', 'gcc', 'ld', 'make')):
    prefix = [] if os.geteuid() == 0 else ['sudo']
    subprocess.run(prefix + ['apt-get', 'update', '-qq'], check=True)
    subprocess.run(prefix + ['apt-get', 'install', '-y', '-qq', 'fasm', 'gcc', 'binutils', 'make'], check=True)
base = Path('/content') if Path('/content').exists() else Path('/tmp')
work = base / 'rulev_variant16' / 'Lab_4'
work.mkdir(parents=True, exist_ok=True)
os.chdir(work)
print('Рабочая папка:', work)

Рабочая папка: /tmp/rulev_variant16/Lab_4


## 3. Общие функции

Число читается с клавиатуры. Цифры переводятся в число последовательным умножением на 10. Для вывода число делится на 10, а полученные цифры записываются в обратном порядке.

### io.inc

In [3]:
%%writefile io.inc
section '.data' writable
    error_text db 'Invalid input', 10
    error_length = $ - error_text

section '.bss' writable
    input_byte rb 1
    output_buffer rb 32

section '.text' executable
read_number:
    xor r8, r8
    xor r9, r9
.read:
    xor eax, eax
    xor edi, edi
    mov rsi, input_byte
    mov edx, 1
    syscall
    test rax, rax
    js invalid_input
    jz .done
    movzx eax, byte [input_byte]
    cmp al, 10
    je .done
    cmp al, '0'
    jb invalid_input
    cmp al, '9'
    ja invalid_input
    sub eax, '0'
    imul r8, 10
    add r8, rax
    cmp r8, r10
    ja invalid_input
    inc r9
    jmp .read
.done:
    test r9, r9
    jz invalid_input
    mov rax, r8
    ret

print_number:
    push rax
    push rbx
    push rcx
    push rdx
    push rsi
    push rdi
    push r11
    mov rsi, output_buffer + 31
    mov byte [rsi], 10
    mov ecx, 1
    mov ebx, 10
.digit:
    xor edx, edx
    div rbx
    add dl, '0'
    dec rsi
    mov [rsi], dl
    inc rcx
    test rax, rax
    jnz .digit
    mov rdx, rcx
    mov eax, 1
    mov edi, 1
    syscall
    pop r11
    pop rdi
    pop rsi
    pop rdx
    pop rcx
    pop rbx
    pop rax
    ret

finish:
    mov eax, 60
    xor edi, edi
    syscall

invalid_input:
    mov eax, 1
    mov edi, 2
    mov rsi, error_text
    mov edx, error_length
    syscall
    mov eax, 60
    mov edi, 1
    syscall


Writing io.inc


### input.h

In [4]:
%%writefile input.h
#ifndef INPUT_H
#define INPUT_H
#include <stdio.h>

static int read_number(long long limit, long long *result) {
    long long value = 0;
    int c;
    int digits = 0;
    while ((c = getchar()) != '\n' && c != EOF) {
        if (c < '0' || c > '9') return 0;
        value = value * 10 + c - '0';
        if (value > limit) return 0;
        digits++;
    }
    if (!digits || ferror(stdin)) return 0;
    *result = value;
    return 1;
}

static int invalid_input(void) {
    fputs("Invalid input\n", stderr);
    return 1;
}
#endif


Writing input.h


## 4. Задание 6

Перебираются числа 5, 10, 15 и так далее. Если число не делится ни на 3, ни на 7, счётчик увеличивается.

### task6.asm

In [5]:
%%writefile task6.asm
format ELF64
public _start

section '.text' executable
_start:
    mov r10, 1000000
    call read_number
    mov r12, rax
    mov ebx, 5
    xor r13, r13
.loop:
    cmp rbx, r12
    ja .done
    mov rax, rbx
    xor edx, edx
    mov ecx, 3
    div rcx
    test rdx, rdx
    jz .next
    mov rax, rbx
    xor edx, edx
    mov ecx, 7
    div rcx
    test rdx, rdx
    jz .next
    inc r13
.next:
    add rbx, 5
    jmp .loop
.done:
    mov rax, r13
    call print_number
    jmp finish

include 'io.inc'


Writing task6.asm


### task6.c

In [6]:
%%writefile task6.c
#include "input.h"

int main(void) {
    long long n;
    long long count = 0;
    if (!read_number(1000000, &n)) return invalid_input();
    for (long long i = 5; i <= n; i += 5) {
        if (i % 3 != 0 && i % 7 != 0) count++;
    }
    printf("%lld\n", count);
    return 0;
}


Writing task6.c


## 5. Задание 8

Функция берёт цифры справа налево и переносит их в разряды 10, 1000, 100000 и так далее. Между цифрами остаются нули. Для n = 0 результат равен 0.

### task8.asm

In [7]:
%%writefile task8.asm
format ELF64
public _start

section '.text' executable
_start:
    mov r10, 999999999
    call read_number
    call insert_zeros
    call print_number
    jmp finish

insert_zeros:
    xor r8, r8
    mov r9, 10
    mov ecx, 10
.loop:
    xor edx, edx
    div rcx
    imul rdx, r9
    add r8, rdx
    test rax, rax
    jz .done
    imul r9, 100
    jmp .loop
.done:
    mov rax, r8
    ret

include 'io.inc'


Writing task8.asm


### task8.c

In [8]:
%%writefile task8.c
#include "input.h"

long long insert_zeros(long long n) {
    long long result = 0;
    long long place = 10;
    do {
        result += n % 10 * place;
        n /= 10;
        if (n != 0) place *= 100;
    } while (n != 0);
    return result;
}

int main(void) {
    long long n;
    if (!read_number(999999999, &n)) return invalid_input();
    printf("%lld\n", insert_zeros(n));
    return 0;
}


Writing task8.c


## 6. Задание 13

Перебираются числа от 10 до n. Выделяются единицы и десятки. Если обе цифры ненулевые и число делится на каждую из них, оно выводится. Однозначные числа не рассматриваются: у них нет двух последних цифр.

### task13.asm

In [9]:
%%writefile task13.asm
format ELF64
public _start

section '.text' executable
_start:
    mov r10, 1000000
    call read_number
    mov r12, rax
    mov ebx, 10
.loop:
    cmp rbx, r12
    ja finish
    mov rax, rbx
    xor edx, edx
    mov ecx, 10
    div rcx
    mov r8, rdx
    xor edx, edx
    div rcx
    mov r9, rdx
    test r8, r8
    jz .next
    test r9, r9
    jz .next
    mov rax, rbx
    xor edx, edx
    div r8
    test rdx, rdx
    jnz .next
    mov rax, rbx
    xor edx, edx
    div r9
    test rdx, rdx
    jnz .next
    mov rax, rbx
    call print_number
.next:
    inc rbx
    jmp .loop

include 'io.inc'


Writing task13.asm


### task13.c

In [10]:
%%writefile task13.c
#include "input.h"

int main(void) {
    long long n;
    if (!read_number(1000000, &n)) return invalid_input();
    for (long long i = 10; i <= n; i++) {
        long long units = i % 10;
        long long tens = i / 10 % 10;
        if (units != 0 && tens != 0 && i % units == 0 && i % tens == 0) {
            printf("%lld\n", i);
        }
    }
    return 0;
}


Writing task13.c


## 7. Сборка

### Makefile

In [11]:
%%writefile Makefile
.PHONY: all check clean

all: build/task6 build/task8 build/task13 build/task6_c build/task8_c build/task13_c

build:
	mkdir -p build

build/%.o: %.asm io.inc | build
	fasm $< $@

build/task%: build/task%.o
	ld -z noexecstack -o $@ $<

build/task%_c: task%.c input.h | build
	gcc -std=c11 -Wall -Wextra -O0 $< -o $@

check: all
	python3 check.py

clean:
	rm -rf build


Writing Makefile


In [12]:
%%bash
set -e
make all

mkdir -p build
fasm task6.asm build/task6.o
flat assembler  version 1.73.32  (16384 kilobytes memory)
5 passes, 1328 bytes.
ld -z noexecstack -o build/task6 build/task6.o
fasm task8.asm build/task8.o
flat assembler  version 1.73.32  (16384 kilobytes memory)
5 passes, 1312 bytes.
ld -z noexecstack -o build/task8 build/task8.o
fasm task13.asm build/task13.o
flat assembler  version 1.73.32  (16384 kilobytes memory)
5 passes, 1344 bytes.
ld -z noexecstack -o build/task13 build/task13.o
gcc -std=c11 -Wall -Wextra -O0 task6.c -o build/task6_c
gcc -std=c11 -Wall -Wextra -O0 task8.c -o build/task8_c
gcc -std=c11 -Wall -Wextra -O0 task13.c -o build/task13_c
rm build/task13.o build/task8.o build/task6.o


## 8. Примеры выполнения

Сначала выводится результат программы на ASM, затем результат программы на C.

In [13]:
%%bash
set -e
printf 'Задание 6, n = 100\n'
printf '100\n' | ./build/task6
printf '100\n' | ./build/task6_c

Задание 6, n = 100
12
12


In [14]:
%%bash
set -e
printf 'Задание 8, n = 123\n'
printf '123\n' | ./build/task8
printf '123\n' | ./build/task8_c
printf 'Задание 8, n = 1005\n'
printf '1005\n' | ./build/task8
printf '1005\n' | ./build/task8_c

Задание 8, n = 123
102030
102030
Задание 8, n = 1005
10000050
10000050


In [15]:
%%bash
set -e
printf 'Задание 13, n = 50, ASM\n'
printf '50\n' | ./build/task13
printf 'Задание 13, n = 50, C\n'
printf '50\n' | ./build/task13_c

Задание 13, n = 50, ASM
11
12
15
22
24
33
36
44
48
Задание 13, n = 50, C
11
12
15
22
24
33
36
44
48


## 9. Проверка

Проверяются обычные числа, нуль, верхние границы и неправильный ввод. Для каждого запуска результат сравнивается с отдельно рассчитанным ответом.

### check.py

In [16]:
%%writefile check.py
from pathlib import Path
import random
import subprocess

root = Path(__file__).resolve().parent
checks = 0


def run(task, text, expected=None):
    global checks
    for suffix in ('', '_c'):
        result = subprocess.run(
            [str(root / 'build' / f'task{task}{suffix}')],
            input=text, text=True, capture_output=True, timeout=10
        )
        if expected is None:
            assert result.returncode == 1, (task, suffix, text, result)
            assert result.stderr == 'Invalid input\n', result.stderr
        else:
            assert result.returncode == 0, (task, suffix, text, result.stderr)
            assert result.stdout == expected, (task, suffix, text, result.stdout, expected)
        checks += 1


rng = random.Random(16)
values = [0, 1, 4, 5, 14, 15, 34, 35, 104, 105, 1000000]
values += [rng.randrange(1000001) for _ in range(20)]
for n in values:
    expected = n // 5 - n // 15 - n // 35 + n // 105
    run(6, f'{n}\n', f'{expected}\n')

values = [0, 1, 9, 10, 100, 101, 123, 100000000, 999999999]
values += [rng.randrange(1000000000) for _ in range(30)]
for n in values:
    expected = int(''.join(digit + '0' for digit in str(n)))
    run(8, f'{n}\n', f'{expected}\n')

values = [0, 9, 10, 11, 12, 20, 22, 99, 100, 101, 110, 1000000]
values += [rng.randrange(10000) for _ in range(15)]
for n in values:
    expected = []
    for value in range(10, n + 1):
        digits = str(value)[-2:]
        if '0' not in digits and all(value % int(digit) == 0 for digit in digits):
            expected.append(f'{value}\n')
    run(13, f'{n}\n', ''.join(expected))

for task in (6, 8, 13):
    limit = 999999999 if task == 8 else 1000000
    for text in ('', '\n', '-1\n', '+1\n', 'abc\n', '12x\n', '1 2\n', f'{limit + 1}\n', '9' * 80 + '\n'):
        run(task, text)
    expected = '10\n' if task == 8 else '0\n' if task == 6 else ''
    run(task, '0001', expected)

print(f'Проверено {checks} запусков. Результаты ASM и C совпадают с проверкой.')


Writing check.py


In [17]:
%%bash
set -e
make check

python3 check.py
Проверено 254 запусков. Результаты ASM и C совпадают с проверкой.


## Вывод

Решены задачи 6, 8 и 13. Ввод в программах на ассемблере выполнен через системный вызов `read`. Результаты программ на ASM и C совпадают с проверочными значениями.